<a href="https://colab.research.google.com/github/jsalafica/Data-Science-III/blob/master/Entrega_Final_DSIII_Javier_Salafica_ok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Entrega Final — NLP y Deep Learning aplicado a textos clínicos

## Introducción

El Procesamiento de Lenguaje Natural (NLP) constituye una herramienta fundamental para el análisis de información clínica no estructurada, como historias médicas, notas de ingreso o descripciones de síntomas. Este tipo de textos, escritos en lenguaje natural, presentan desafíos particulares debido a su variabilidad lingüística, uso de terminología médica y ambigüedad semántica.

El objetivo de este trabajo es construir un **pipeline completo de NLP en español** que permita clasificar textos clínicos simples según el **servicio médico más probable**, abordando el problema como una tarea de **clasificación supervisada single-label**. Para ello, se aplican técnicas de preprocesamiento del lenguaje, representación vectorial de textos y entrenamiento de modelos predictivos.

Se desarrollan y comparan dos enfoques principales:
- un **modelo clásico de Machine Learning**, basado en TF-IDF y Regresión Logística,
- y una **red neuronal sencilla (Deep Learning)** implementada con Keras.

Ambos modelos reciben como entrada texto clínico libre y devuelven una predicción del servicio médico asociado, permitiendo analizar similitudes, diferencias y limitaciones entre enfoques tradicionales y neuronales cuando se trabaja con representaciones textuales.

El dataset utilizado consiste en un archivo CSV con aproximadamente **1000 registros clínicos**, cargado desde un repositorio en GitHub. Cada registro contiene una descripción textual de síntomas y un único servicio médico asociado. Dado que los textos pueden incluir información ambigua o multisistémica, el trabajo también explora un enfoque alternativo de **recomendación de servicios**, basado en la distribución de probabilidades de los modelos, como apoyo a la decisión clínica.

In [ ]:
# Instalación de dependencias NLP
!pip -q install spacy
!python -m spacy download es_core_news_sm -q
!pip -q install wordcloud

In [115]:
## Imports
import re
import numpy as np
import pandas as pd

import spacy
from spacy.lang.es.stop_words import STOP_WORDS

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from tensorflow import keras
from tensorflow.keras import layers

from wordcloud import WordCloud


## 1. Carga del dataset

Se carga el dataset desde GitHub (formato CSV, separador `;`).  
El dataset contiene las columnas:

- `texto`: texto clínico en español  
- `servicio`: etiqueta (single-label)


In [ ]:
##URL = "https://raw.githubusercontent.com/jsalafica/Data-Science-III/refs/heads/master/dataset_nlp_1200.csv"
URL = "https://raw.githubusercontent.com/jsalafica/Data-Science-III/refs/heads/master/dataset_singlelabel_1000.csv"

df = pd.read_csv(URL, sep=";")

print(df.shape)
df.head()

In [ ]:
print(df.columns)
print(df["servicio"].value_counts().head(10))
print("N clases:", df["servicio"].nunique())
print("N nulos texto:", df["texto"].isna().sum())
print("N nulos servicio:", df["servicio"].isna().sum())


## 2. Preprocesamiento NLP

Se aplica preprocesamiento con **spaCy**:

- Normalización (minúsculas y limpieza básica)
- Tokenización y lematización
- Eliminación de stopwords
- Preservación de información etaria mediante tokens semánticos:
  - `edad_adulto`
  - `edad_pediatrico`
  - `edad_neonatal`


In [ ]:
nlp = spacy.load("es_core_news_sm")
stop_es = set(STOP_WORDS)

len(stop_es)


Durante el preprocesamiento se eliminaron términos altamente frecuentes y no discriminativos, como “paciente”, que no aportan información categórica para la clasificación. Esta decisión permitió reducir ruido en el vocabulario y mejorar la interpretabilidad del análisis exploratorio de texto.


In [138]:
STOPWORDS_CLINICAS = {
    "paciente",
    "ingresa",
    "consulta",
    "refiere",
    "presenta"
}

In [139]:
def preprocess_spacy(texto: str) -> str:
    texto = str(texto).lower()
    texto = re.sub(r"\s+", " ", texto).strip()

    # ---- Preservación etaria CORREGIDA (evita semanas de evolución) ----

    # Detecta edad SOLO si no está seguida por "evolucion"
    pat_edad = re.compile(
        r"\b(\d{1,3})\s*(años?|mes(?:es)?|d[ií]as?|horas?|semanas?)\b(?!\s+de\s+evoluci[oó]n)",
        flags=re.IGNORECASE
    )

    def _edad_a_dias(valor: int, unidad: str) -> int:
        unidad = unidad.lower()
        if unidad.startswith("año"):
            return valor * 365
        if unidad.startswith("mes"):
            return valor * 30
        if unidad.startswith("semana"):
            return valor * 7
        if unidad.startswith("día") or unidad.startswith("dia"):
            return valor
        if unidad.startswith("hora"):
            return 0
        return None

    def _clasificar_edad_por_dias(dias: int) -> str:
        if dias < 30:
            return "edad_neonatal"
        if dias < (16 * 365):
            return "edad_pediatrico"
        return "edad_adulto"

    def _reemplazar_edad(match):
        valor = int(match.group(1))
        unidad = match.group(2)
        dias = _edad_a_dias(valor, unidad)
        if dias is None:
            return match.group(0)
        return f" {_clasificar_edad_por_dias(dias)} "

    texto = pat_edad.sub(_reemplazar_edad, texto)


    # ---- spaCy ----
    doc = nlp(texto)

    tokens = []
    for tok in doc:
        if tok.is_space or tok.is_punct:
            continue

        lemma = tok.lemma_.strip()
        if not lemma:
            continue

        # eliminar números puros (edad ya fue tokenizada)
        if lemma.isdigit():
            continue

        if lemma in stop_es:
            continue

        # eliminar stopwords estándar y clínicas
        if lemma in STOPWORDS_CLINICAS:
            continue

        tokens.append(lemma)

    return " ".join(tokens)


In [ ]:
df["texto_proc"] = df["texto"].astype(str).apply(preprocess_spacy)

df[["texto", "texto_proc"]].head(10)


### Nube de palabras

Se generó una nube de palabras a partir del texto preprocesado, lo cual permite
visualizar los términos más frecuentes del dataset y obtener una primera
aproximación exploratoria al contenido textual.


In [ ]:
texto_total = " ".join(df["texto_proc"].dropna().astype(str).tolist())

wc = WordCloud(
    width=1400,
    height=800,
    background_color="white",
    collocations=False  # evita juntar bigramas raros
).generate(texto_total)

plt.figure(figsize=(14, 8))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Mapa de palabras (dataset completo)")
plt.show()


In [ ]:
def wordcloud_por_servicio(servicio, max_words=200):
    subset = df.loc[df["servicio"] == servicio, "texto_proc"].dropna().astype(str)
    texto = " ".join(subset.tolist())

    wc = WordCloud(
        width=1400,
        height=800,
        background_color="white",
        max_words=max_words,
        collocations=False
    ).generate(texto)

    plt.figure(figsize=(14, 8))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Mapa de palabras — {servicio}")
    plt.show()

# Ejemplo:
wordcloud_por_servicio("UTI")


## 3. Vectorización (TF-IDF)

Transformamos el texto preprocesado a una matriz numérica usando TF-IDF.
Se utilizan unigramas y bigramas.


In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=8000
)

X = vectorizer.fit_transform(df["texto_proc"])
y = df["servicio"]

print("X shape:", X.shape)
print("Clases:", y.nunique())


## 4. Modelo clásico (Regresión Logística)

Se entrena un modelo base de clasificación multiclase usando Regresión Logística.
Se divide el dataset en train/test 80/20 con estratificación por clase.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


## 5. Análisis de resultados (Profundización NLP)

Se analiza la matriz de confusión para observar clases que se confunden entre sí.


In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels, normalize="true")

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
fig, ax = plt.subplots(figsize=(12, 12))
disp.plot(ax=ax, xticks_rotation=90, values_format=".2f", colorbar=True)
plt.title("Matriz de confusión normalizada (por clase real)")
plt.tight_layout()
plt.show()


In [ ]:
cm_raw = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm_raw, index=labels, columns=labels)

cm_err = cm_df.copy()
np.fill_diagonal(cm_err.values, 0)

top = cm_err.stack().sort_values(ascending=False).head(15)

print("Top 15 confusiones (Real -> Predicho : cantidad):")
for (real, pred), cnt in top.items():
    if cnt > 0:
        print(f"{real} -> {pred}: {cnt}")


## 6. Interpretabilidad del modelo

La Regresión Logística permite inspeccionar los términos con mayor peso por clase.
Además, se muestra una explicación simple de una predicción individual.


In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
classes = model.classes_
coef = model.coef_

def top_terms_for_class(class_idx, top_n=12):
    top_pos_idx = np.argsort(coef[class_idx])[-top_n:][::-1]
    return list(zip(feature_names[top_pos_idx], coef[class_idx][top_pos_idx]))

for i, cls in enumerate(classes):
    tops = top_terms_for_class(i, top_n=10)
    print(f"\n=== {cls} | Top términos ===")
    for term, w in tops:
        print(f"{term:25s} {w:.3f}")


In [ ]:
def explain_prediction(texto_original, top_k=12):
    texto_p = preprocess_spacy(texto_original)
    X_one = vectorizer.transform([texto_p])

    scores = model.decision_function(X_one).ravel()
    pred_idx = np.argmax(scores)
    pred_class = model.classes_[pred_idx]

    row = X_one.tocoo()
    contrib = {}
    for j, v in zip(row.col, row.data):
        contrib[j] = contrib.get(j, 0.0) + (coef[pred_idx, j] * v)

    top = sorted(contrib.items(), key=lambda kv: kv[1], reverse=True)[:top_k]
    top_terms = [(feature_names[j], float(val)) for j, val in top]

    return {"texto_proc": texto_p, "prediccion": pred_class, "top_terms": top_terms}

ejemplo = "Paciente de 46 años con antecedente mencionados consulta por cuadro de 2 semanas de evolución caracterizado por cefalea holocraneana de intensidad 7/10 sin irradiación que se asocia a sensación febril y dos episodios de vómitos de contenido gástrico no precedidos por nauseas. Al interrogatorio dirigido refiere presentar de 1 mes de evolución, diarrea acuosa no disenteriforme, tos seca y disfagia. Ingresa a Clínica médica para diagnóstico y tratamiento"
info = explain_prediction(ejemplo, top_k=12)

print("Texto procesado:", info["texto_proc"])
print("Predicción:", info["prediccion"])
print("Top términos que empujaron la predicción:")
for t, v in info["top_terms"]:
    print(f"{t:25s} {v:.4f}")


## Recomendación de múltiples servicios (enfoque Top-k / umbral)

Si bien el modelo fue entrenado como un clasificador *single-label* (una sola especialidad por texto), en escenarios clínicos reales los cuadros suelen ser complejos y pueden involucrar más de un servicio potencial.

Con el objetivo de reflejar esta realidad sin modificar el esquema de entrenamiento, se incorporó una función de **recomendación de servicios** basada en la distribución de probabilidades del clasificador.

### Metodología
- Se utilizan las probabilidades estimadas por el modelo para cada clase.
- Se devuelven:
  - **Top-k** servicios más probables (ranking).
  - Servicios cuya probabilidad supere un **umbral configurable**.

Este enfoque no constituye una clasificación multietiqueta real, pero permite interpretar la salida del modelo como un **sistema de apoyo a la decisión**, mostrando alternativas plausibles en lugar de forzar una única predicción.

### Interpretación
- En textos claros y poco ambiguos, un único servicio suele dominar el ranking.
- En textos complejos o multisistémicos, la probabilidad se distribuye entre varios servicios, reflejando la ambigüedad inherente al lenguaje clínico.
- La ausencia de servicios por encima del umbral indica **incertidumbre del modelo**, lo cual resulta clínicamente razonable.

Este procedimiento mejora la utilidad práctica del modelo y permite analizar sus limitaciones sin alterar el dataset ni el entrenamiento original.


In [ ]:
def recomendar_servicios(texto, top_k=5, threshold=0.15):
    t = preprocess_spacy(texto)
    v = vectorizer.transform([t])
    probs = model.predict_proba(v)[0]
    pares = sorted(zip(model.classes_, probs), key=lambda x: x[1], reverse=True)

    top = [(c, float(p)) for c, p in pares[:top_k]]
    por_umbral = [(c, float(p)) for c, p in pares if p >= threshold]

    return {"top_k": top, "por_umbral": por_umbral}

texto_ejemplo = ("Paciente de 46 años con cuadro de dos semanas de evolución caracterizado por cefalea holocraneana, vómitos, sensación febril y diarrea acuosa. Presenta tos seca y disfagia. Ingresa para diagnóstico y tratamiento.")

recomendar_servicios(
    texto_ejemplo
)



## 7. Deep Learning (Red neuronal simple)

Se entrena una red neuronal feed-forward sencilla usando como entrada la matriz TF-IDF.


In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(y)
num_classes = len(le.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

X_train_d = X_train.toarray()
X_test_d  = X_test.toarray()

y_train_oh = keras.utils.to_categorical(y_train, num_classes=num_classes)
y_test_oh  = keras.utils.to_categorical(y_test,  num_classes=num_classes)

X_train_d.shape, y_train_oh.shape, num_classes


In [ ]:
input_dim = X_train_d.shape[1]

model_nn = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

model_nn.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model_nn.summary()


In [ ]:
early = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model_nn.fit(
    X_train_d, y_train_oh,
    validation_split=0.2,
    epochs=15,
    batch_size=32,
    callbacks=[early],
    verbose=1
)


In [ ]:
y_pred_prob = model_nn.predict(X_test_d)
y_pred = np.argmax(y_pred_prob, axis=1)

print("Accuracy DL:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))


In [ ]:
def predict_nn(texto: str, top_k=5):
    t = preprocess_spacy(texto)
    v = vectorizer.transform([t]).toarray()
    p = model_nn.predict(v)[0]
    idx = np.argsort(p)[::-1][:top_k]
    return [(le.classes_[i], float(p[i])) for i in idx]

predict_nn("Paciente de 46 años con cuadro de dos semanas de evolución caracterizado por cefalea holocraneana, vómitos, sensación febril y diarrea acuosa. Presenta tos seca y disfagia. Ingresa para diagnóstico y tratamiento")

## Recomendación de múltiples servicios con Deep Learning

Además de la predicción clásica de una única especialidad, se implementó una función de recomendación basada en la salida probabilística de la red neuronal (`predict_nn`). Este enfoque permite interpretar el modelo como un sistema de apoyo a la decisión clínica, especialmente útil en textos complejos o multisistémicos.

### Metodología
- La red neuronal devuelve una probabilidad para cada servicio mediante una capa *softmax*.
- A partir de estas probabilidades se construye:
  - un **ranking Top-k** de servicios más probables,
  - y un filtrado por **umbral mínimo de probabilidad**.

Este procedimiento no constituye una clasificación multietiqueta real, dado que el entrenamiento fue *single-label*, pero permite analizar la incertidumbre del modelo y considerar múltiples alternativas plausibles.

### Interpretación de la salida
- Cuando un servicio concentra gran parte de la probabilidad, el texto suele ser claro y específico.
- Cuando la probabilidad se distribuye entre varias clases, el modelo refleja ambigüedad inherente al lenguaje clínico.
- La ausencia de servicios por encima del umbral indica incertidumbre del modelo, lo cual resulta clínicamente razonable y preferible a una predicción forzada.

Este enfoque mejora la interpretabilidad del modelo sin modificar el dataset ni el esquema de entrenamiento original.


In [ ]:
def recomendar_servicios_nn(texto, top_k=5, threshold=0.15):
    """
    Recomendador multi-servicio (Top-k / Umbral) usando la red neuronal nn/model_nn.
    Requiere: preprocess_spacy, vectorizer, nn (modelo Keras), y le (LabelEncoder).
    """
    # Preprocesar igual que en el pipeline
    t = preprocess_spacy(texto)
    v = vectorizer.transform([t]).toarray()

    # Probabilidades softmax
    probs = model_nn.predict(v, verbose=0)[0]  # shape: (n_clases,)
    pares = sorted(zip(le.classes_, probs), key=lambda x: x[1], reverse=True)

    top = [(c, float(p)) for c, p in pares[:top_k]]
    por_umbral = [(c, float(p)) for c, p in pares if p >= threshold]

    return {"texto_procesado": t,"top_k": top, "por_umbral": por_umbral}

texto = "presenta hipotensión grave que requiere vasopresores, insuficiencia respiratoria aguda con necesidad de ventilación mecánica, fiebre, alteración del estado de conciencia y fallo renal agudo"
recomendar_servicios_nn(texto)


## 8. Conclusiones

En este trabajo se desarrolló un sistema de clasificación de textos clínicos en español utilizando técnicas de Procesamiento de Lenguaje Natural (NLP) y modelos de Machine Learning y Deep Learning. Se aplicaron distintas etapas de preprocesamiento, incluyendo normalización lingüística, lematización y preservación semántica de la edad del paciente, lo que permitió reducir ruido y mejorar la coherencia clínica de los textos analizados.

Se entrenaron y compararon modelos clásicos basados en TF-IDF y Regresión Logística con una red neuronal sencilla, observándose que, para este tipo de representación textual, ambos enfoques presentan desempeños comparables. Al utilizar un dataset más heterogéneo y cercano a escenarios reales, el modelo mostró un accuracy moderado pero realista, evidenciando dificultades en clases con muy pocos ejemplos y buen rendimiento en especialidades con mayor representación.

Asimismo, se incorporó un enfoque de recomendación de servicios basado en rankings de probabilidad, que permite interpretar la salida del modelo como una herramienta de apoyo a la decisión en lugar de una clasificación determinística. Esta estrategia resulta especialmente útil en textos clínicos complejos, donde la asignación a un único servicio puede no reflejar adecuadamente la práctica médica real.

La eliminación de términos altamente frecuentes y no discriminativos permitió reducir ruido en la representación TF-IDF. Como consecuencia, el modelo mejoró su capacidad de generalización al concentrar el aprendizaje en tokens clínicamente relevantes, lo que se reflejó en una mejora de las métricas de desempeño.

Finalmente, se identificaron limitaciones inherentes al enfoque utilizado, tales como la falta de razonamiento numérico y la dependencia exclusiva del lenguaje explícito. Como líneas de trabajo futuro se propone la incorporación de clasificación multietiqueta real, el uso de variables clínicas estructuradas y el entrenamiento con datasets más balanceados, lo que permitiría mejorar la robustez y aplicabilidad del sistema.
